# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import VotingClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split

from feature_engine.encoding import RareLabelEncoder
from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

In [3]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)

def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Dataset

In [4]:
X_train = pd.read_parquet('../data/X_train.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

In [5]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.298107,0.207392,0.252153,0.279021,0.298107,0.298107,0.321361,0.252153,0.279021,0.253721
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.298107,0.185508,0.252153,0.586119,0.298107,0.298107,0.610176,0.252153,0.544023,0.613159
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.298107,0.207392,0.252153,0.207676,0.298107,0.298107,0.423519,0.252153,0.228194,0.231339
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.102471,0.185508,0.178575,0.138352,0.102471,0.102471,0.043550,0.178575,0.138352,0.070991
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.153276


# Machine Learning

In [6]:
lgbm = load_pickle('../models/model_lightgbm.pkl')
cat = load_pickle('../models/model_catboostencoder_catboost.pkl')
xgb = load_pickle('../models/model_xgboost.pkl')
hist = load_pickle('../models/model_histgradientboosting.pkl')

In [7]:
voting = VotingClassifier(
    estimators=[
        ('lgbm', lgbm), 
        ('cat', cat), 
        ('xgb', xgb), 
        ('hist', hist)
    ],
    voting='soft',
    weights=np.power([0.9483091624611604, 0.9464863062095595, 0.9485412032495437, 0.9485286479852023], 2)
)

voting.fit(X_train, y_train.PitNextLap)

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/externals/_numpydoc/docscrape.py:420: UserWarning: Unknown section Example
  self[section] = content


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('lgbm', ...), ('cat', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.","array([0.8992..., 0.8997066 ])"
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,features_to_drop,"['driver_avg_position', 'laps_since_pit', ...]"
,boosting_type,'gbdt'
,num_leaves,251
,max_depth,9
,learning_rate,0.024925863277580798


# Deploy

In [8]:
dump_pickle(voting, '../models/model_voting.pkl')